In [14]:
import os
import pandas as pd
from obspy import read, UTCDateTime
from datetime import timedelta

# ============ KONFIGURASI PATH ============
input_root = r"E:\Skripsi\DEC\0-data2025\MEPAC_HHZ"
csv_path = r"E:\Skripsi\DEC\0-data2025\bulletin_edit_test10.xlsx"
output_root = r"E:\Skripsi\DEC\0-data2025\filtered_events_noice10"

# ============ BACA DATAFRAME EVENT ============
print("Membaca file buletin...")
events = pd.read_excel(csv_path)
events.columns = events.columns.str.strip()

# Konversi waktu
events['eventdate'] = pd.to_datetime(events['eventdate'], format='%d/%m/%Y %H:%M:%S')
events['event_datetime_utc'] = events['eventdate'] - timedelta(hours=7)

# Bersihkan eventtype untuk nama folder
events['eventtype'] = events['eventtype'].astype(str).str.upper().str.strip().str.replace(' ', '_')

print(f"Jenis event yang ditemukan: {events['eventtype'].unique()}")

# ============ PROSES FILE MSEED ============
mseed_files = [f for f in os.listdir(input_root) if f.endswith('.mseed')]
print(f"\nMenemukan {len(mseed_files)} file MSEED")

for mseed_file in mseed_files:
    file_path = os.path.join(input_root, mseed_file)
    print(f"\nMemproses: {mseed_file}")
    
    # Baca data MSEED
    st = read(file_path)
    tr = st[0]
    start_time = tr.stats.starttime
    end_time = tr.stats.endtime
    
    # Filter event dalam rentang waktu
    mask = (
        (events['event_datetime_utc'] >= start_time.datetime) & 
        (events['event_datetime_utc'] <= end_time.datetime)
    )
    
    relevant_events = events[mask]
    
    if relevant_events.empty:
        print("Tidak ada event dalam rentang waktu ini.")
        continue
    
    print(f"Menemukan {len(relevant_events)} event")
    
    # Proses setiap event
    for _, ev in relevant_events.iterrows():
        ev_time = UTCDateTime(ev['event_datetime_utc'])
        duration = float(ev['duration'])
        event_type = ev['eventtype']
        
        # Waktu potong
        t1 = ev_time
        t2 = ev_time + duration 
        
        # Buat folder sesuai event type
        event_folder = os.path.join(output_root, event_type)
        os.makedirs(event_folder, exist_ok=True)
        
        # Potong data
        tr_cut = tr.copy().trim(starttime=t1, endtime=t2)
        
        # Format nama file
        ev_time_str = ev_time.strftime('%Y-%m-%d_%H-%M-%S')
        output_name = f"{ev_time_str}-{event_type}.mseed"
        output_path = os.path.join(event_folder, output_name)
        
        # Simpan file
        tr_cut.write(output_path, format='MSEED')
        print(f"  ✓ {output_name}")

print(f"\nSelesai! File disimpan di: {output_root}")

Membaca file buletin...
Jenis event yang ditemukan: ['ROCKFALL' 'MP' 'VTB' 'TECT' 'LF' 'AWANPANAS' 'UNKNOWN']

Menemukan 16 file MSEED

Memproses: VG.MEPAC.00.HHZ.D.2025.257.mseed
Menemukan 48 event
  ✓ 2025-09-14_17-14-24-ROCKFALL.mseed
  ✓ 2025-09-14_17-25-55-MP.mseed
  ✓ 2025-09-14_17-39-46-ROCKFALL.mseed
  ✓ 2025-09-14_17-40-49-MP.mseed
  ✓ 2025-09-14_17-55-57-MP.mseed
  ✓ 2025-09-14_18-00-08-MP.mseed
  ✓ 2025-09-14_18-02-38-ROCKFALL.mseed
  ✓ 2025-09-14_18-08-41-VTB.mseed
  ✓ 2025-09-14_18-15-34-TECT.mseed
  ✓ 2025-09-14_18-31-15-MP.mseed
  ✓ 2025-09-14_18-39-17-MP.mseed
  ✓ 2025-09-14_18-41-21-ROCKFALL.mseed
  ✓ 2025-09-14_18-50-39-ROCKFALL.mseed
  ✓ 2025-09-14_18-59-43-ROCKFALL.mseed
  ✓ 2025-09-14_19-11-15-MP.mseed
  ✓ 2025-09-14_19-16-06-ROCKFALL.mseed
  ✓ 2025-09-14_19-39-42-ROCKFALL.mseed
  ✓ 2025-09-14_19-45-02-ROCKFALL.mseed
  ✓ 2025-09-14_19-50-59-MP.mseed
  ✓ 2025-09-14_20-03-03-ROCKFALL.mseed
  ✓ 2025-09-14_20-11-10-MP.mseed
  ✓ 2025-09-14_20-13-54-MP.mseed
  ✓ 2025-09-